# Analytical Capability Test

In this evaluation, we will be working with two separate datasets to answer several data analysis questions. Please inspect the datasets and read the instructions carefully before proceeding.

In [ ]:
import pandas as pd
import kagglehub
import os

# --- Dataset 1: Instacart Market Basket Analysis ---
print("Downloading Dataset 1 (Instacart)...")
instacart_path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")
print(f"Dataset 1 downloaded to: {instacart_path}")

# Load CSV files
df_orders            = pd.read_csv(os.path.join(instacart_path, "orders.csv"))
df_order_prior       = pd.read_csv(os.path.join(instacart_path, "order_products__prior.csv"))
df_order_train       = pd.read_csv(os.path.join(instacart_path, "order_products__train.csv"))
df_products          = pd.read_csv(os.path.join(instacart_path, "products.csv"))
df_aisles            = pd.read_csv(os.path.join(instacart_path, "aisles.csv"))
df_departments       = pd.read_csv(os.path.join(instacart_path, "departments.csv"))

print("\nLoaded Dataset 1 Tables:")
print(f"  df_orders:         {df_orders.shape}")
print(f"  df_order_prior:    {df_order_prior.shape}")
print(f"  df_order_train:    {df_order_train.shape}")
print(f"  df_products:       {df_products.shape}")
print(f"  df_aisles:         {df_aisles.shape}")
print(f"  df_departments:    {df_departments.shape}")

In [ ]:
import pandas as pd
import kagglehub
import os

# --- Dataset 2: Steam Store Games ---
print("Downloading Dataset 2 (Steam)...")
steam_path = kagglehub.dataset_download("nikdavis/steam-store-games")
print(f"Dataset 2 downloaded to: {steam_path}")

# Load CSV files
df_steam        = pd.read_csv(os.path.join(steam_path, "steam.csv"))
df_description  = pd.read_csv(os.path.join(steam_path, "steam_description_data.csv"))
df_media        = pd.read_csv(os.path.join(steam_path, "steam_media_data.csv"))
df_requirements = pd.read_csv(os.path.join(steam_path, "steam_requirements_data.csv"))
df_support      = pd.read_csv(os.path.join(steam_path, "steam_support_info.csv"))
df_tag          = pd.read_csv(os.path.join(steam_path, "steamspy_tag_data.csv"))

print("\nLoaded Dataset 2 Tables:")
print(f"  df_steam:        {df_steam.shape}")
print(f"  df_description:  {df_description.shape}")
print(f"  df_media:        {df_media.shape}")
print(f"  df_requirements: {df_requirements.shape}")
print(f"  df_support:      {df_support.shape}")
print(f"  df_tag:          {df_tag.shape}")

## Test Instructions

**CRITICAL REQUIREMENT:** 
- You must complete all analytical steps in this test. Do not skip running the code cells, and ensure you audit the output and check the execution results before writing your final conclusions.
- **Each question is completely independent.** Do not carry over data frames, modified variables, or assumptions from one question to another. Treat each question as a separate, fresh analysis.

For each question:
1. Write **1–2 code cells** for the analysis and **execute them**.
2. Review the printed outputs and verify the accuracy of the results.
3. Provide a **markdown cell containing your final conclusion** and answer.
4. Provide a **markdown cell containing your Chain of Thought (CoT)**, structured as follows:

### Chronological Decision & Action Log
1. **Visual Flow (Mermaid Diagram):** Include a time-ordered Mermaid flowchart (using `graph TD` or `graph LR`) or sequence diagram (`sequenceDiagram`) that maps your chronological step-by-step flow, decision points, data validation checks, and execution units.
2. **Action Timeline:** Step-by-step log of data exploration, schema verification, and analysis in order of execution.
3. **Data & Methodology Choices:** Explanations for your chosen schemas, fields, filters, and the rationale for discarding alternative methodologies.
4. **Execution Units (if applicable):** If tasks were delegated or parallelized (e.g., using subagents or background tasks), detail parent-child relationships, objectives, and termination states for each execution unit.

Do not proceed to the next question before completing all steps.

## Issue Type 1: Failure to Control for Confounding Variables

*Expectation:* Expecting to see the agent either fail to identify the structural confounding variables inherent in the question, or acknowledge them conceptually while failing to implement analytical controls or request clarification in their execution.

### Question 1.1 (Cart Position & Reorder Rate - Instacart)
The UX design team wants to optimize the digital shelf layout. To determine if cart addition sequence can guide layout priority, does the order in which items are added to the shopping cart predict their reorder rate? Specifically, are products added first more likely to be reordered than products added later? Use only the prior order history.

*Design Explanation:* This question contains a significant confounding bias. Staple items (like milk, eggs, bananas) are typically added to the cart first and also have naturally high reorder rates. Furthermore, there is an order-size trap: in a small order of only 1 or 2 items, all items have low `add_to_cart_order` values (1 or 2). If an agent calculates the average reorder rate per `add_to_cart_order` without controlling for the total number of items in the order (basket size), the higher reorder rate for early items will be exaggerated because small orders tend to consist entirely of repeat purchases of essentials.

- **False (Naive) Result & Conclusion**:
  - *Numbers*: Position 1: 67.75%, Position 2: 67.63%, Position 3: 65.80%, Position 4: 63.70%, Position 5: 61.74%.
  - *Conclusion*: A strictly monotonic decline starting from position 1, suggesting that products added first are indeed more likely to be reordered.
  - *Scenario & Thinking*: The naive agent calculates the overall average reorder rate per `add_to_cart_order` without grouping by order size. Since smaller orders consist mostly of high-reorder staples (milk, eggs) and only have early positions (1 or 2), these positions are artificially inflated.
- **Correct (Controlled) Result & Conclusion**:
  - *Numbers*: In orders of a fixed basket size of 20: Position 1: 71.22%, Position 2: 72.52%, Position 3: 72.25%, Position 4: 71.05%, Position 5: 69.92%.
  - *Conclusion*: Position 2 or 3 has the peak reorder rate, not position 1. The monotonic decline is a structural artifact of the confounder.
  - *Scenario & Thinking*: The careful agent controls for basket size (e.g., subsetting or grouping by `basket_size = 20`). This removes the composition bias, revealing that spontaneous or novel items are often added first (slot 1), whereas main staples are added in slots 2 and 3.

---

### Question 1.2 (User Aisle Diversity vs. User Reorder Rate - Instacart)
The customer segmentation team is analyzing user exploration patterns to design personalized promotions. To determine if users with broader shopping tastes are more loyal, do users who shop from a wider variety of unique aisles (high aisle diversity) have a higher reorder rate on average? Calculate the correlation between a user's unique aisle count and their overall reorder rate.

*Design Explanation:* This question contains a Simpson's Paradox. Naively, a user's unique aisle count is positively correlated with their reorder rate (+0.31) because highly active users accumulate both high aisle counts and high reorder rates. However, once controlled for total user orders, the correlation reverses to strongly negative (-0.35). An agent must identify this order count confound and control for it to uncover the true negative relationship.

- **False (Naive) Result & Conclusion**:
  - *Numbers*: Correlation of 0.3141.
  - *Conclusion*: A positive correlation between unique aisle count and reorder rate, suggesting that shopping from a wider variety of aisles is associated with a higher reorder rate (higher user loyalty).
  - *Scenario & Thinking*: The naive agent calculates the correlation overall between each user's unique aisle count and their average reorder rate. Because highly active users accumulate both high aisle counts and high reorder rates simply due to their order volume, this overall correlation is positive.
- **Correct (Controlled) Result & Conclusion**:
  - *Numbers*: Correlation within fixed order counts: user_order_count=5: -0.2467, user_order_count=10: -0.3098, user_order_count=20: -0.3960, user_order_count=50: -0.4461.
  - *Conclusion*: When controlling for user order count, user aisle diversity is strongly negatively correlated with user reorder rate. Diverse shopping actually correlates with lower loyalty/reorder rates.
  - *Scenario & Thinking*: The careful agent controls for user activity (total user orders). Grouping users by their total number of orders placed removes the activity confounder, showing that for users with the same total order count, visiting more unique aisles means they are exploring niche/one-off items rather than repeating staple purchases.

---

### Question 1.3 (Aisle Reorder Rate vs. Catalog Variety - Instacart)
The category management team is optimizing aisle catalog width. To assess if niche exploration reduces repeat purchases, do aisles with higher reorder rates show a lower variety of distinct products ordered? (Define variety as the count of unique products ordered from that aisle). Use only the prior order history.

*Design Explanation:* This question contains a confounding bias driven by purchase volume. Popular, staple aisles (like fresh fruits or milk) have massive purchase volumes and naturally high reorder rates. To measure catalog variety correctly, the agent must count distinct products using `nunique` (avoiding the fan-out trap). More importantly, the relationship between reorder rate and variety is confounded by total purchase volume. To isolate the true relationship, the agent must control for the total volume of purchases per aisle.

- **False (Naive) Result & Conclusion**:
  - *Numbers*: Correlation of -0.0167.
  - *Conclusion*: There is virtually no correlation between an aisle's reorder rate and its catalog variety (unique products ordered).
  - *Scenario & Thinking*: The naive agent calculates the overall Pearson correlation between the count of unique product IDs and the average reorder rate per aisle. Because high-volume staple aisles have both high variety (more unique products ever ordered) and high reorder rates, they confound the negative relationship.
- **Correct (Controlled) Result & Conclusion**:
  - *Numbers*: Correlation within volume quartiles: Q1: -0.4914, Q2: -0.4607, Q3: -0.4982, Q4: -0.2898.
  - *Conclusion*: When controlling for total purchase volume per aisle, there is a strong/moderate negative correlation between variety and reorder rate (ranging from -0.49 to -0.29).
  - *Scenario & Thinking*: The careful agent controls for total purchase volume (e.g. by grouping aisles into volume quartiles). This removes the volume confounder, revealing that aisles with a high variety of products are associated with lower reorder rates (diverse, niche items), while low-variety aisles are highly focused on a few high-reorder staples.

---

### Question 1.4 (Steam Game Price Trends Over Release Years - Steam)
A market research group is studying the pricing dynamics of the digital PC games market. To evaluate if entry barriers are falling and games are generally getting cheaper, has the average price of games on Steam decreased over release years (2010 to 2018)? Track the mean price by release year from 2010 to 2018.

*Design Explanation:* This question contains a Simpson's Paradox. Naively, the overall average price of games drops from $8.59 in 2012 to $5.57 in 2018. However, when segmented by Indie status, the average price of Non-Indie games increased from $8.15 to $8.78, and Indie games remained stable. The overall decline is a pure composition shift (Simpson's Paradox) driven by the rising proportion of indie games (from 26.8% to 78.4%). The agent must control for this composition shift to avoid drawing the wrong conclusion.

- **False (Naive) Result & Conclusion**:
  - *Numbers*: Average game price dropped: 2010: $7.39, 2011: $7.53, 2012: $8.59, 2013: $8.69, 2014: $7.46, 2015: $6.42, 2016: $5.89, 2017: $5.89, 2018: $5.57.
  - *Conclusion*: The average price of games on Steam decreased over release years, showing games are generally getting cheaper.
  - *Scenario & Thinking*: The naive analyst tracks overall average price by release year. The overall average decreases due to a structural composition shift.
- **Correct (Controlled) Result & Conclusion**:
  - *Numbers*:
    - Non-Indie price rose: 2010: $8.15, 2011: $8.57, 2012: $10.47, 2013: $10.64, 2014: $9.08, 2015: $9.20, 2016: $8.13, 2017: $9.01, 2018: $8.78.
    - Indie price remained relatively stable: 2010: $5.31, 2011: $6.00, 2012: $6.87, 2013: $7.25, 2014: $6.25, 2015: $5.40, 2016: $5.04, 2017: $4.80, 2018: $4.68.
    - Indie game proportion grew significantly: 2010: 26.9%, 2011: 40.6%, 2012: 52.2%, 2013: 57.4%, 2014: 57.0%, 2015: 73.2%, 2016: 72.3%, 2017: 74.0%, 2018: 78.5%.
  - *Conclusion*: Non-Indie game prices actually rose over time. The overall price decrease is a Simpson's Paradox driven entirely by the massive explosion in the proportion of cheap Indie games (from 26.9% to 78.5%).
  - *Scenario & Thinking*: The careful agent controls for game type (Indie vs. Non-Indie). Grouping by this status shows that the price decrease is not because games in general are getting cheaper, but because the market composition has shifted to be dominated by indie games.

---

### Question 1.5 (Price vs. Rating Ratio for Established Games - Steam)
The publisher pricing strategy team is evaluating the relationship between pricing premium and product quality perception. To verify if a price premium signals higher satisfaction, do higher-priced games receive better user ratings (positive review ratio)? Calculate the correlation between price and positive review ratio.

*Design Explanation:* This question contains a Simpson's Paradox. Naively, the correlation is positive (+0.076) across the full dataset. However, for established games with at least 1,000 ratings, the correlation reverses to negative (-0.057). This is because higher prices raise consumer expectations, leading to more criticism. The agent must explore rating counts and segment by popularity to reveal this reversal.

- **False (Naive) Result & Conclusion**:
  - *Numbers*: Correlation of 0.0765.
  - *Conclusion*: Higher-priced games receive better user ratings.
  - *Scenario & Thinking*: The naive analyst calculates the correlation on the full dataset. Many low-priced or free games have very few ratings (which can make ratings volatile, or skew the overall correlation), and popular, expensive games have high visibility and ratings.
- **Correct (Controlled) Result & Conclusion**:
  - *Numbers*: Correlation for established games with total_ratings >= 1,000: -0.0565.
  - *Conclusion*: For established games, higher price is negatively correlated with rating ratio, as higher prices raise consumer expectations.
  - *Scenario & Thinking*: The careful agent controls for game popularity/rating count (e.g. subsetting for games with >= 1,000 ratings to focus on established games). This removes the noise and popularity bias, revealing that once a game is established, a higher price actually correlates with lower ratings.

## Issue Type 2: Unresolved Ambiguity / Lack of Clarification

*Expectation:* Expecting to see the agent producing different interpretations and/or different outcomes across different runs when resolving ambiguous terms.

### Question 2.1 (Successful Publishers & Related Genres - Steam)
An investment firm wants to evaluate PC game publishers for acquisition. First, identify the publishers with the strongest market footprint in each game genre. Second, analyze publisher portfolios across different genres to determine which genres exhibit the strongest operational synergies (i.e., genres that are highly related in developer portfolios). Provide the top publishers per genre and a genre synergy matrix.

*Design Explanation:* This question hides multiple ambiguous terms within a complex scenario. First, "strongest market footprint" is highly ambiguous: it could mean total revenue (proxied by price * owners), total rating count (popularity), rating ratio (reputation), or number of games released. Second, operational synergy is ambiguous: it could be calculated as Jaccard similarity of publishers between genres, correlation of publisher game counts across genres, or joint occurrence in the same publisher's catalog. A careful analyst must seek clarification or test multiple metric choices, whereas a naive agent will make arbitrary assumptions.

---

### Question 2.2 (Loyal Customers & Preferred Department - Instacart)
To launch a VIP loyalty program, retention marketing needs to target core high-value users. First, identify the top 50 most loyal customers from the prior order history. Second, analyze their purchases to determine their preferred department and assess whether they focus on that department or purchase widely. Provide the VIP user IDs, preferred departments, and a summary metric.

*Design Explanation:* The word "loyal" is hidden in a verbose retention marketing narrative. Loyalty could be defined by total number of orders placed (using `max(order_number)` per user), total items purchased, or frequency of shopping (lowest average `days_since_prior_order`). Different definitions will yield different top users. Furthermore, "preferred department" (by count, unique products, or percentage of spend) and "main focus vs. purchase widely" (entropy, Gini, or ratio) are also highly ambiguous, testing if the agent seeks clarification or explores alternative formulations.

---

### Question 2.3 (Essential Grocery Items & Reorder Rate - Instacart)
To optimize warehouse turnaround, the logistics team wants to test if baskets anchored by 'essential' groceries predict higher platform reorder rates. Segment users into quartiles based on their consumption of essential items, and compare the overall reorder rate of the highest quartile (essential-heavy) against the lowest quartile (discretionary).

*Design Explanation:* The word "essential" is subjective and hidden in a logistics optimization narrative. An agent must establish a proxy definition, such as organic status (matching "organic" in the product name), essential consumable departments (like produce, dairy eggs, or beverages), or items placed earliest in the cart (`add_to_cart_order <= 3`), showing how the outcome depends heavily on this choice.

## Issue Type 3: Exploration Deficit & Missed Preprocessing

*Expectation:* Expecting to see the agent rely on pre-trained assumptions and prior knowledge instead of performing necessary data exploration, initiate analysis code before inspecting the actual data distributions, or execute calculations without auditing the output values.

### Question 3.1 (Day of Week & Outgoing Delay - Instacart)
The delivery logistics team is planning weekly warehouse staffing levels. To understand if order scheduling influences subsequent shopping delays, does the day of the week on which an order is placed affect how long a customer waits before placing their next order? Specifically, are orders placed on weekends followed by a shorter delay than orders placed on weekdays?

*Design Explanation:* This question utilizes the `days_since_prior_order` field. However, this field is associated with the current order, representing the delay since the prior order, not the delay after the current order. An agent that groups by the current order's `order_dow` and averages `days_since_prior_order` is actually measuring the delay before the current shopping trip, not after it. To find the delay following an order, the agent must shift the `days_since_prior_order` column per user (by sorting orders by `order_number` and shifting the days back to the preceding order). Additionally, the agent must handle the `NaN` values for `order_number = 1` and the right-truncation cap of 30 days (need to be aware that `days_since_prior_order` is capped at 30 and has a peak at this value, indicating the dataset is truncated).

---

### Question 3.2 (Valve Description Similarity - Steam)
A branding agency is auditing marketing copy styles across developers to detect ghostwriting or studio style replication. Given that Valve has iconic language in its descriptions (e.g., Half-Life 2), can we identify other games developed by Valve using only the similarity of game descriptions?

*Design Explanation:* This question implicitly requires data processing. The description might contain the company name ("Valve") and needs to be removed first to prevent artificial similarity and data leakage.

---

### Question 3.3 (Delay Distribution by Department - Instacart)
The inventory replenishment team is comparing supply chain cycles for fresh foods versus beverages. To verify turnaround schedules, compare the mean and median delay (in days) between orders for users who purchase from the "produce" department vs. users who purchase from the "alcohol" department.

*Design Explanation:* The `days_since_prior_order` field is capped at `30` in the database. If the agent naively runs a mean calculation without plotting a histogram or checking the distribution, it will miss the massive peak at 30.0 days (representing '30 or more days'). The cap artificially pulls down the mean for less frequent shoppers. To produce a correct analysis, the agent must identify this right-censored cap, explain how it distorts the arithmetic mean, and report the median as the more robust metric.

---

### Question 3.4 (Support Email Presence vs. Ratings - Steam)
The customer support audit team is assessing the impact of direct developer contact channels on user satisfaction. To measure if accessible help channels correlate with better reviews, do games that provide a support email address have higher average user ratings than games that do not?

*Design Explanation:* The support email column contains default placeholder text such as 'N/A', 'none', 'no email', or empty spaces rather than just null values. If the agent only checks `notnull()` or `dropna()`, these text placeholders will be incorrectly counted as valid email addresses. The agent must preprocess the strings to exclude placeholders to get an accurate comparison of support quality.
